### 从Tbot-base中生成BPVA policy

In [1]:
# 独立单元：从 TBot base、已有 BPVA 权重或随机初始化构造 BPVA policy。
# 通常只需修改 BP_NUM_CHUNKS、BP_ACTION_CHUNK_SIZE 和 SAVE_DIR。
from pathlib import Path
import torch

from lerobot.policies.BPVA.configuration_bpva import BPVAConfig
from lerobot.policies.BPVA.modeling_bpva import BPVAPolicy

LOAD_MODE = "TBOT_BASE"  # 可选："TBOT_BASE" / "BPVA" / "SCRATCH"
TBOT_BASE_DIR = Path("/vla/workspace/models/tbot_base")
BPVA_SOURCE_DIR = Path("/vla/workspace/models/bpva_init_c10_a50")
SAVE_DIR = Path("/vla/workspace/models/bpva_init_c10_a50")

QWEN3_VL_DIR = Path("/vla/workspace/models/Qwen3-VL-2B-Instruct")
COSMOS_DIR = Path("/vla/workspace/models/Cosmos-Tokenizer-CI8x8")
DEVICE = "cuda"
DTYPE = "bfloat16"
STRICT_LOAD = False
ALLOW_OVERWRITE = False

# K：每个样本包含的 behavior prompt chunk 数量。
BP_NUM_CHUNKS = 10
# 每个 behavior prompt chunk 中包含的 action 时间步数。
BP_ACTION_CHUNK_SIZE = 50

# 下列 BPObsEncoder 结构参数一般保持不变。
BP_VISION_MODEL_NAME = "vit_base_patch16_clip_224.openai"
BP_VISION_PRETRAINED = True
BP_FREEZE_VISION_ENCODER = True
BP_TOKEN_DIM = 768
BP_IMAGE_FEATURE_AGGREGATION = "cls"

# 初始化 checkpoint 不加载 DA3 teacher；正式训练可通过训练配置重新启用 3D loss。
ENABLE_3D_QUERIES = True
LAMBDA_3D = 0.0


# 直接用最终参数构造 config，确保与 tools/generate_bpva_init_checkpoint.py 一致。
cfg = BPVAConfig(
    pretrained_path=None,
    device=DEVICE,
    dtype=DTYPE,
    qwen3_vl_pretrained_path=str(QWEN3_VL_DIR),
    cosmos_tokenizer_path_or_name=str(COSMOS_DIR),
    bp_num_chunks=BP_NUM_CHUNKS,
    bp_action_chunk_size=BP_ACTION_CHUNK_SIZE,
    bp_vision_model_name=BP_VISION_MODEL_NAME,
    bp_vision_pretrained=BP_VISION_PRETRAINED,
    bp_freeze_vision_encoder=BP_FREEZE_VISION_ENCODER,
    bp_token_dim=BP_TOKEN_DIM,
    bp_image_feature_aggregation=BP_IMAGE_FEATURE_AGGREGATION,
    enable_3d_queries=ENABLE_3D_QUERIES,
    lambda_3d=LAMBDA_3D,
)


/vla/.conda/miniconda3/envs/mytbot/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
if LOAD_MODE == "TBOT_BASE":
    LOAD_DIR = TBOT_BASE_DIR
    policy_init = BPVAPolicy.from_pretrained(LOAD_DIR, config=cfg, strict=STRICT_LOAD)
elif LOAD_MODE == "BPVA":
    LOAD_DIR = BPVA_SOURCE_DIR
    policy_init = BPVAPolicy.from_pretrained(LOAD_DIR, config=cfg, strict=STRICT_LOAD)
elif LOAD_MODE == "SCRATCH":
    LOAD_DIR = None
    policy_init = BPVAPolicy(cfg)
else:
    raise ValueError(f"不支持的 LOAD_MODE：{LOAD_MODE}")

policy_init.eval()
print("LOAD_MODE:", LOAD_MODE)
print("LOAD_DIR:", LOAD_DIR)
print("SAVE_DIR:", SAVE_DIR)
print("policy name:", policy_init.name)
print("device:", cfg.device, "dtype:", cfg.dtype)
print("bp_obs_encoder in state_dict:", any(k.startswith("model.bp_obs_encoder.") for k in policy_init.state_dict()))
print("num state_dict keys:", len(policy_init.state_dict()))

Loading weights from local directory


LOAD_MODE: TBOT_BASE
LOAD_DIR: /vla/workspace/models/tbot_base
SAVE_DIR: /vla/workspace/models/bpva_init_c10_a50
policy name: bpva
device: cuda dtype: bfloat16
bp_obs_encoder in state_dict: True
num state_dict keys: 1963


In [3]:
if LOAD_MODE == "TBOT_BASE":
    if SAVE_DIR.exists() and any(SAVE_DIR.iterdir()) and not ALLOW_OVERWRITE:
        raise FileExistsError(f"输出目录非空：{SAVE_DIR}。请修改 SAVE_DIR 或设置 ALLOW_OVERWRITE=True。")
    # 独立单元：保存 policy 权重与 config.json，并验证 BPObsEncoder 参数确实写入。
    # 依赖上一单元得到的 policy_init / SAVE_DIR；重启 kernel 后请先运行上一单元。
    from pathlib import Path
    from huggingface_hub.constants import SAFETENSORS_SINGLE_FILE
    from safetensors import safe_open

    SAVE_DIR = Path(SAVE_DIR)
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    # 使用公开保存接口，写入 config.json 和 model.safetensors。
    policy_init.save_pretrained(SAVE_DIR)

    config_path = SAVE_DIR / "config.json"
    weight_path = SAVE_DIR / SAFETENSORS_SINGLE_FILE
    if not config_path.is_file() or not weight_path.is_file():
        raise RuntimeError(f"checkpoint 保存不完整：{SAVE_DIR}")

    with safe_open(weight_path, framework="pt", device="cpu") as checkpoint:
        keys = list(checkpoint.keys())
    bp_keys = [key for key in keys if key.startswith("model.bp_obs_encoder.")]
    if not bp_keys:
        raise RuntimeError("保存的 checkpoint 中没有 model.bp_obs_encoder.* 权重")

    print("saved to:", SAVE_DIR)
    print("files:", sorted(path.name for path in SAVE_DIR.iterdir()))
    print("config type:", policy_init.config.type)
    print("num saved keys:", len(keys))
    print("num bp_obs_encoder keys:", len(bp_keys))
    print("first bp key:", bp_keys[0])
    print(f"weight size: {weight_path.stat().st_size / 1024**3:.2f} GiB")


saved to: /vla/workspace/models/bpva_init_c10_a50
files: ['config.json', 'model.safetensors']
config type: bpva
num saved keys: 1458
num bp_obs_encoder keys: 170
first bp key: model.bp_obs_encoder.chunk_encoder.action_proj.0.bias
weight size: 6.53 GiB


### 检查模型可推理性

In [4]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATASET_PATH = '/vla/workspace/data/robotwin2.0/hanging_mug/aloha-agilex_clean_50'
ACTION_CHUNK_SIZE = 50
FPS = 30

current_delta_timestamps = {
    "action": [i / FPS for i in range(ACTION_CHUNK_SIZE)],
    "observation.images.cam_high": [-0.5, 0.0, 0.5],
    "observation.images.cam_left_wrist": [-0.5, 0.0, 0.5],
    "observation.images.cam_right_wrist": [-0.5, 0.0, 0.5],
}
# prompt_ds：图像只取当前关键帧，action 仍返回完整未来窗口。
prompt_delta_timestamps = {
    "action": [i / FPS for i in range(ACTION_CHUNK_SIZE)],
}
current_ds = LeRobotDataset(DATASET_PATH, delta_timestamps=current_delta_timestamps)
prompt_ds = LeRobotDataset(DATASET_PATH, delta_timestamps=prompt_delta_timestamps)


In [5]:
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptLeRobotDataset, BehaviorPromptConfig

config = BehaviorPromptConfig()
config.prompt_action_chunk_size = ACTION_CHUNK_SIZE
config.max_prompt_chunks = None
config.num_chunks = 10
config.same_episode_policy = "avoid"
config.seed = 0

bp_ds = BehaviorPromptLeRobotDataset.with_default_transforms(current_ds, prompt_ds, config)
sample = bp_ds[0]
for i, step in enumerate(bp_ds.transform.transforms):
    print(f"data process step:  [{i}] {step.__class__.__name__}")
print("BP state:", sample["behavior_prompt"]["state"].shape)
print("BP action:", sample["behavior_prompt"]["action"].shape)
bp_ds

Hydrating transform InjectMissingStateActionTransformFn (robot_type=aloha, resolved=aloha, action_seq_len=1, state_seq_len=1, placeholder_dim=14)
Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform DeltaActionTransformFn with mapping and mask (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)
data process step:  [0] BPRemapImageKeyTransformFn
data process step:  [1] BPPadOrSampleChunksFn
data process step:  [2] BPResizeImagesWithPadFn
data process step:  [3] BPComposeFieldsTransform
data process step:  [4] BPDeltaActionTransformFn
data process step:  [5] BPNormalizeTransformFn
data process step:  [6] BPPadStateAndActionTransformFn
data process step:  [7] InjectMissingStateActionTransformFn

In [6]:
from torch.utils.data import DataLoader
from torch.utils.data._utils.collate import default_collate
# 用bp_ds[0],bp_ds[1] 构建batch size =2 的batch
samples = [bp_ds[0],bp_ds[1]]
batch = default_collate(samples)

import torch
def move_to_device(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device)
    if isinstance(x, dict):
        return {k: move_to_device(v, device) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to_device(v, device) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to_device(v, device) for v in x)
    return x
batch_for_forward = move_to_device(batch, "cuda")

In [7]:
# 加载模型 - 可选
if  LOAD_MODE== 'TBOT_BASE':
    policy = BPVAPolicy.from_pretrained(SAVE_DIR, config=cfg, strict=STRICT_LOAD)
elif LOAD_MODE == 'BPVA':
    policy = policy_init
else:
    raise ValueError(f"暂时不支持的 LOAD_MODE：{LOAD_MODE}")

Loading weights from local directory


In [ ]:
# 真实 batch 前向：返回 total loss 和各项 loss 日志。
# 注意：这是正式模型规模，显存压力明显高于前面的 BPObsEncoder 单模块验证。
torch.cuda.empty_cache()
policy.train()
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    loss, loss_dict = policy(batch_for_forward)

print("loss:", float(loss.detach().cpu()))
for key, value in loss_dict.items():
    if key.startswith("loss_action_dim"):
        continue
    print(f"{key}: {value}")

# 反向也走一下，确认 BPObsEncoder 在同一个计算图里。
loss.backward()
print("backward ok")

loss: 0.21227207779884338
loss: 0.21227207779884338
loss_action: 0.19298779964447021
loss_gen: 1.9284272193908691
loss_3d: 0.0
backward ok


: 